In [1]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
import plotly.graph_objects as go

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from CRUD import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "MLP2002"

# Connect to database via CRUD Module
db = AnimalShelter()

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))
rescue_types = ['Water Rescue', 'Mountain or Wilderness Rescue', 'Disaster or Individual Tracking', 'Reset']

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#Add in Grazioso Salvare’s logo
image_filename = 'Grazioso-Salvare-Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode('utf-8')

app.layout = html.Div([
    # html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Center(html.Img(src='data:image/png;base64,{}'.format(encoded_image), style={'width': '200px', 'height': 'auto'})),
    html.Hr(),
    html.Center(html.B(html.H3('Jaiden Blanchard'))),
    html.Div(
        className='animalButtonRow', 
        style={'display': 'flex'},
        children=[
            # filter for animal types
            html.Button(id='submit-button-one', n_clicks=0, children='Cats'),
            html.Button(id='submit-button-two', n_clicks=0, children='Dogs'),
            html.Button(id='submit-button-three', n_clicks=0, children='Birds'),
            html.Button(id='submit-button-four', n_clicks=0, children='Other'),
        ]
    ), 
    html.Hr(),
    html.Div(
    # filter for rescue types
    dcc.RadioItems(
            id='filter-type',
            options=[{'label': i, 'value': i} for i in rescue_types],
            value='Reset',
            labelStyle={'display': 'inline-block', 'padding': '10px', 'border': '1px solid #ccc', 'borderRadius': '5px', 'margin': '5px'}
        ), 
    ),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        editable=True,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable="single",
        row_selectable="single",
        row_deletable=True,
        selected_columns=[],
        selected_rows=[],
        style_cell={'textAlign': 'left'},
        page_action="native",
        page_current=0,
        page_size=10
    ),
    html.Br(),
    html.Hr(),
    # Graph for breed data
    html.H4('Animals'),
    dcc.Graph(id='graph'),
    html.Br(),
    html.Hr(),
    # This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row', style={'display': 'flex'},
        children=[
            html.Div(id='graph-id', className='col s12 m6'),
            html.Div(id='map-id', className='col s12 m6')
        ]
    )
])
        
#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('submit-button-one', 'n_clicks'),
     Input('submit-button-two', 'n_clicks'),
     Input('submit-button-three', 'n_clicks'),
     Input('submit-button-four', 'n_clicks'),
     Input('filter-type', 'value')])

def animal_click(catButton, dogButton, birdButton, otherButton, filter_type):
    # Create a list of button clicks and determine which is clicked the most
    button_clicks = [catButton, dogButton, birdButton, otherButton]
    animal_types = {0: "Cat", 1: "Dog", 2: "Bird", 3: "Other"}

    max_button_index = max(range(len(button_clicks)), key=lambda i: button_clicks[i])
    query = {"animal_type": animal_types[max_button_index]} if button_clicks[max_button_index] > 0 else {}

    # fetch data based on the button and clean up ID field
    df = pd.DataFrame.from_records(db.read(query))
    df.drop(columns=['_id'], inplace=True, errors='ignore')

    # Apply filtering based on the rescue type
    filter_criteria = {
    'Water Rescue': {
        'breeds': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland'],
        'sex': 'Intact Female',
        'age_range': (26, 156)
    },
    'Mountain or Wilderness Rescue': {
        'breeds': ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler'],
        'sex': 'Intact Male',
        'age_range': (26, 156)
    },
    'Disaster or Individual Tracking': {
        'breeds': ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever', 'Bloodhound', 'Rottweiler'],
        'sex': 'Intact Male',
        'age_range': (20, 300)
    }
}

    # Apply the filter based on filter_type
    if filter_type == 'Reset':
        dff = df
    else:
        criteria = filter_criteria.get(filter_type)
        if criteria:
            dff = df[
                df.breed.isin(criteria['breeds']) &
                (df.sex_upon_outcome == criteria['sex']) &
                (df.age_upon_outcome_in_weeks >= criteria['age_range'][0]) &
                (df.age_upon_outcome_in_weeks <= criteria['age_range'][1])
            ]
        else:
            dff = df

    return dff.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table

@app.callback(
    Output("graph", "figure"), 
    Input('datatable-id', "derived_virtual_data"))

def update_graphs(viewData):
    if viewData is None:
        return go.Figure()  # Return an empty figure if no data

    dff = pd.DataFrame.from_dict(viewData)
    breed_counts = dff['breed'].value_counts()
    
    fig = px.bar(breed_counts, x=breed_counts.index, y=breed_counts.values, title='Animal Breed Total')
    return fig
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns'),
    Input('datatable-id', 'selected_rows')]
)   
def update_styles(selected_columns, selected_rows):
    styles = []
        # Apply styles for selected rows
    if selected_rows is not None:
        styles.extend([
            {'if': {'row_index': row},
             'backgroundColor': 'lightblue'
            } for row in selected_rows
        ])

    # Apply styles for selected columns
    if selected_columns is not None:
        styles.extend([
            {'if': {'column_id': col},
             'backgroundColor': 'lightblue'
            } for col in selected_columns
        ])
    return styles


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def display_map(viewData, selected_rows):
    return update_map(viewData, selected_rows)

def update_map(viewData, index):    
    dff = pd.DataFrame.from_dict(viewData)
     # Because we only allow single row selection, the list can 
     # be converted to a row index here
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Ensure that the row index is within the bounds of the DataFrame
    if row >= len(dff):
        row = 0 
    
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[30.75, -97.48], zoom=10, children=[
                   dl.TileLayer(id="base-layer-id"),
                   dl.Marker(position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                              children=[
                                  dl.Tooltip(dff.iloc[row, 4]),
                                  dl.Popup([
                                      html.P(dff.iloc[row, 9])
                                  ])
                              ])
                           ])
                        ]

app.run_server(debug=True)

Dash app running on http://127.0.0.1:30871/
